# Summary

Create a JSONL file with IPEDS table metadata

In [1]:
import json
import re
import os
import datetime

#### Numantic utilities
utils_path = "../../../Utilities/text_cleaning"
sys.path.insert(0, utils_path)
import text_cleaning_tools as tct
import json_tools as jt


In [3]:
def parse_single_metadata_block(block: str) -> dict:
    # Extract fields using regex
    table_name_match = re.search(r"BigQuery table name:\s*(.+?)\.", block, re.IGNORECASE)
    overview_match = re.search(r"Overview description of file contents:\s*(.+?)\.\s*Data columns:", block, re.IGNORECASE | re.DOTALL)
    data_dict_match = re.search(r"Data dictionary:\s*(.+)", block, re.IGNORECASE | re.DOTALL)

    if not (table_name_match and overview_match and data_dict_match):
        # Skip incomplete blocks
        return None

    table_name = table_name_match.group(1).strip()
    overview = overview_match.group(1).strip()

    raw_data_dict = data_dict_match.group(1).strip()
    raw_data_dict = raw_data_dict.rstrip(". ")

    # Parse data dictionary
    data_dictionary = {}
    entries = raw_data_dict.split(":::,")
    for entry in entries:
        parts = entry.strip().split("::")
        if len(parts) >= 2:
            col_name = parts[0].strip()
            col_desc = "::".join(parts[1:]).strip().rstrip(":::.")
            if col_name:
                data_dictionary[col_name] = col_desc

    return {
        "BigQuery table name": table_name,
        "Overview description of file contents": overview,
        "Data dictionary": data_dictionary
    }

def parse_multiple_metadata_blocks(filepath: str, output_dir: str):
    with open(filepath, "r", encoding="utf-8") as file:
        content = file.read()

    # Split content by "BigQuery table name:" but keep the delimiter to each block by adding it back
    blocks = re.split(r"(?=BigQuery table name:)", content)

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for block in blocks:
        metadata = parse_single_metadata_block(block)
        if metadata:
            table_name = metadata["BigQuery table name"]
            filename = f"{table_name}.json"
            filepath_out = os.path.join(output_dir, filename)
            with open(filepath_out, "w", encoding="utf-8") as f_out:
                json.dump(metadata, f_out, indent=4)
            print(f"Saved metadata for table '{table_name}' to {filepath_out}")

## Read crawl parameters

In [4]:
crawl_parameters_file = "crawl_parameters.json"
crawl_data_path = "../data/crawler_params"
os.listdir(crawl_data_path)


with open(os.path.join(crawl_data_path, crawl_parameters_file), "r") as file:
    cpdat = json.load(file)

# Find the IPEDS entry
for cp in cpdat:
    if cp["storage_folder"] == "ipeds":
        ipeds_cpdat = cp

# Create an IPEDS organization for the JSON file
ipeds_org = dict(name=ipeds_cpdat["organization"],
                 role=ipeds_cpdat["about"],
                 uri=ipeds_cpdat["seed_url"])
ipeds_org



{'name': 'Integrated Postsecondary Education Data System',
 'role': 'IES is the independent research, statistics, and evaluation arem of the U.S. Department of Education.IPEDS is a system of 12 interrelated survey components conducted annually postsecondary institutions participating in federal student financial aid programs.',
 'uri': 'https://nces.ed.gov/ipeds'}

## Read in IPEDS text metadata

In [5]:
input_file = "ipeds_tables_descriptions_2025Apr29.txt"
input_path = "../data/ipeds"
with open(os.path.join(input_path, input_file), "r", encoding="utf-8") as file:
    file_content = file.read()

blocks = re.split(r"(?=BigQuery table name:)", file_content)



## Create a list of dictionaries with data to convert to JSON

In [6]:
# Set common parameters
available_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
uri = ipeds_cpdat["seed_url"]
source_index = ipeds_cpdat["storage_folder"]

txt_lines = []
for i, blk_txt in enumerate(blocks):

    blk = parse_single_metadata_block(blk_txt)

    if blk and len(blk) > 0:

        content = ("Overview description of file contents: {}\n\n"
            "Data dictionary: {}").format(blk["Overview description of file contents"],
                                          json.dumps(blk["Data dictionary"]))

        # Clean the content
        content = tct.clean_web_texts(web_texts=[content])

        # Add a to a list of dictionaries to create JSONL files
        txt_lines.append(dict(title="Description of IPEDS table: {}".format(blk["BigQuery table name"]),
                              description="web page text",
                              source_index=source_index,
                              media_type="text",
                              language_code="en-US",
                              categories=["education > text"],
                              uri=uri,
                              country_origin="US",
                              content_index=i,
                              transcript=content,
                              organizations=[ipeds_org],
                              available_time=available_time
                          )
         )



## Save a JSONL file

In [7]:
Jsonl_outfile = "crawl_data_schema_jsonl_ipeds_metadata.jsonl"
out_path = "../data/jsonl_files"
os.listdir(out_path)

jt.create_json_lines_file(data=txt_lines,
                          filename=os.path.join(out_path, Jsonl_outfile))
